# FedSAA revision experiments (reviewer items 1–6)

Runs everything the reviewer asked for, on **CPU**, from cached frozen-backbone features:

| Item | What runs |
|---|---|
| 1 | 10 seeds × all strategies × 200 rounds; metrics saved at rounds 40, 100 and 200 |
| 2 | FedSAA-U (retention in update space) and FedSAA with ρ = 0, next to the original FedSAA |
| 3 | Specialty-oracle clustered FL, FlexLoRA, FFA-LoRA, FedSA-LoRA, and a linear probe |
| 4a | Nonlinear adapter h = x + s·B·ReLU(Ax) with local, centralized, FedAvg, FedProx, oracle, FFA-LoRA and FedSA-LoRA |
| 5 | Similarity matrices, attention weights and the retained SVD energy are logged; per-image correctness is saved |
| 6 | Learning rate {3e-4, 1e-3, 3e-3} (and FedProx μ {0.001, 0.01, 0.1}) chosen on a validation split |

**Settings → Internet: On. Accelerator: None (CPU).**

1. Optional smoke test: set `QUICK = True` and run all cells (about 10 minutes, mostly the download). Then set it back to `False`.
2. Real run: **Save Version → Save & Run All (Commit)**. It runs in the background, so you can close the browser; the estimate is 6–7 hours, within Kaggle's 12-hour limit.
3. When it finishes, open the version's **Output** tab, download `fedbench_v2_results.zip` and upload it to the chat.

To resume after a run that stopped early, attach that version's output as input (*Add Input → Your Work*) and set `RESUME_FROM` to its `results` folder; finished runs are skipped.

Item 4b (LoRA inside the convolution layers) needs a GPU and is not part of this notebook.

In [ ]:
QUICK = False          # True = smoke test on a few rounds and 3 seeds
RESUME_FROM = ""       # e.g. "/kaggle/input/<previous-version>/results" to resume
TUNE_BUDGET_MIN = 150  # stop tuning after this many minutes (rerun to continue)
MAIN_BUDGET_MIN = 540  # stop the main runs after this many minutes (rerun to continue)
import os
if QUICK:
    os.environ["FEDBENCH_QUICK"] = "1"
else:
    os.environ.pop("FEDBENCH_QUICK", None)

In [ ]:
!pip install -q medmnist

In [ ]:
%%writefile fedbench_features.py
#!/usr/bin/env python3
"""Stage 1 (run once, on Kaggle): build the clients and cache frozen ResNet-18 features.

Usage:
    python fedbench_features.py --out feats [--root medmnist_data]

For each preset this writes feats/preset_A.npz and feats/preset_B.npz holding, for every client,
the 256-d features after the third residual stage (global-average-pooled) of the train, validation and
test images, their labels, and a JSON metadata record.

Partition (fixed, independent of the training seed, so every strategy and seed sees the same clients):
  * official MedMNIST train split, shuffled with PARTITION_SEED, cut into 3 equal contiguous parts;
    each part capped at 2,000 images; the first 15% of each part becomes that client's validation set;
  * official test split, shuffled, cut into 3 equal parts, each capped at 1,000 images.
Images are the native 64 x 64 MedMNIST+ files, converted to 3 channels and normalized with the
ImageNet mean and standard deviation. No augmentation is used, which is what makes caching valid.
"""
import argparse
import json
import os
import time

import numpy as np
import torch
import torchvision

PRESETS = {
    "A": ["pathmnist", "dermamnist", "bloodmnist", "octmnist", "tissuemnist"],
    "B": ["organamnist", "pneumoniamnist", "retinamnist", "breastmnist", "organcmnist"],
}
CLIENTS_PER_SPECIALTY = 3
TRAIN_CAP, TEST_CAP, VAL_FRAC = 2000, 1000, 0.15
PARTITION_SEED = 2026
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def load_split(flag, split, root):
    import medmnist
    from medmnist import INFO
    cls = getattr(medmnist, INFO[flag]["python_class"])
    ds = cls(split=split, download=True, size=64, root=root)
    imgs = np.asarray(ds.imgs)
    labels = np.asarray(ds.labels).reshape(-1).astype(np.int64)
    return imgs, labels, len(INFO[flag]["label"])


def to_tensor(imgs):
    x = imgs.astype(np.float32) / 255.0
    if x.ndim == 3:
        x = np.repeat(x[..., None], 3, axis=-1)
    x = (x - MEAN) / STD
    return torch.from_numpy(x.transpose(0, 3, 1, 2).copy())


def make_backbone(device):
    m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
    body = torch.nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool, m.layer1, m.layer2, m.layer3,
                               torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten())
    return body.eval().to(device)


@torch.no_grad()
def extract(body, imgs, device, bs=512):
    out = []
    for k in range(0, len(imgs), bs):
        out.append(body(to_tensor(imgs[k:k + bs]).to(device)).float().cpu().numpy())
    return np.concatenate(out).astype(np.float32)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="feats")
    ap.add_argument("--root", default="medmnist_data")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    os.makedirs(args.root, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.set_num_threads(max(1, os.cpu_count() or 1))
    body = make_backbone(device)
    rng = np.random.default_rng(PARTITION_SEED)
    for preset, flags in PRESETS.items():
        t0 = time.time()
        arrays, meta = {}, {"preset": preset, "specialties": flags, "client_specialty": [], "n_classes": [],
                            "n_train": [], "n_val": [], "n_test": [], "partition_seed": PARTITION_SEED,
                            "train_cap": TRAIN_CAP, "test_cap": TEST_CAP, "val_frac": VAL_FRAC,
                            "image_size": 64, "feature": "resnet18_layer3_gap_256"}
        c = 0
        for s, flag in enumerate(flags):
            tr_x, tr_y, k = load_split(flag, "train", args.root)
            te_x, te_y, _ = load_split(flag, "test", args.root)
            tr_parts = np.array_split(rng.permutation(len(tr_y)), CLIENTS_PER_SPECIALTY)
            te_parts = np.array_split(rng.permutation(len(te_y)), CLIENTS_PER_SPECIALTY)
            for tp, sp in zip(tr_parts, te_parts):
                tp, sp = tp[:TRAIN_CAP], sp[:TEST_CAP]
                nva = int(round(VAL_FRAC * len(tp)))
                va, tr = tp[:nva], tp[nva:]
                for name, idx, X, Y in (("tr", tr, tr_x, tr_y), ("va", va, tr_x, tr_y), ("te", sp, te_x, te_y)):
                    arrays[f"X{name}_{c}"] = extract(body, X[idx], device)
                    arrays[f"y{name}_{c}"] = Y[idx]
                meta["client_specialty"].append(s)
                meta["n_classes"].append(int(k))
                meta["n_train"].append(int(len(tr)))
                meta["n_val"].append(int(len(va)))
                meta["n_test"].append(int(len(sp)))
                c += 1
            print(f"  preset {preset}: {flag} done ({k} classes)", flush=True)
        meta["n_clients"] = c
        np.savez_compressed(os.path.join(args.out, f"preset_{preset}.npz"), meta=json.dumps(meta), **arrays)
        print(f"preset {preset}: {c} clients, train {sum(meta['n_train'])}, val {sum(meta['n_val'])}, "
              f"test {sum(meta['n_test'])}, {time.time() - t0:.0f} s", flush=True)
        print("  test sizes:", meta["n_test"], flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile fedbench_train.py
#!/usr/bin/env python3
"""Stage 2: train every strategy on the cached features (pure NumPy, CPU-friendly, resumable).

Usage:
    python fedbench_train.py --feats feats --out results --stage tune --budget-min 40
    python fedbench_train.py --feats feats --out results --stage main --budget-min 40
Run each stage repeatedly until it prints "stage complete"; finished runs are skipped.

Model on the cached 256-d feature x of client i:
    linear adapter     h = x + s B A x            (s = alpha / r)
    nonlinear adapter  h = x + s B relu(A x)
    no adapter         h = x                      (linear probe)
    logits = W_i h + c_i, with a private classifier per client (classes of other specialties masked).
All clients of a run share one initialization: A ~ N(0, 1/d), B = 0, W ~ N(0, 0.01^2), c = 0.
Each round is E local epochs with Adam; the optimizer state is reset at the start of every round for
every strategy. All 15 clients take part in every round.

Strategies (server step after each round; classifiers are never aggregated):
    local       no communication
    centralized one shared adapter and one classifier per specialty, trained on the pooled data;
                each step averages the gradients of one mini-batch from every client
    fedavg      sample-weighted average of A and of B
    fedprox     fedavg + proximal term mu/2 ||theta - theta_global||^2 on the adapter
    oracle      fedavg restricted to the clients of the same specialty (specialty-oracle clustered FL)
    flexlora    sample-weighted average of the products BA, rank-r truncated SVD, same factors for all
    ffa         FFA-LoRA: A frozen at its (shared) initialization, B trained and averaged
    fedsa       FedSA-LoRA: A averaged, B kept local
    fedsaa      FedSAA, retention in factor space (rho = 0.3)
    fedsaa_r0   FedSAA without retention (rho = 0)
    fedsaa_u    FedSAA-U, retention in update space: (1-rho) U~_i + rho U_i, then one truncated SVD
    probe       local, no adapter (linear probe)
Product-space strategies (flexlora, fedsaa*) are defined for the linear adapter only.
"""
import argparse
import json
import math
import os
import time

import numpy as np

# ------------------------------------------------------------------ configuration
RANK, ALPHA = 16, 32.0
LOCAL_EPOCHS, BATCH = 3, 64
ROUNDS, CHECKPOINTS = 200, (40, 100, 200)
LOG_ROUNDS = (1, 10, 40, 100, 200)
TAU, LAM, RHO = 0.5, 0.5, 0.3
LR_GRID = (3e-4, 1e-3, 3e-3)
MU_GRID = (1e-3, 1e-2, 1e-1)
TUNE_SEED = 1000
TUNE_ROUNDS, TUNE_CHECKPOINTS = 100, (40, 100)
MAIN_SEEDS = tuple(range(10))
if os.environ.get("FEDBENCH_QUICK"):            # smoke test only
    ROUNDS, CHECKPOINTS, LOG_ROUNDS = 6, (2, 4, 6), (1, 6)
    TUNE_ROUNDS, TUNE_CHECKPOINTS, MAIN_SEEDS = 4, (2, 4), tuple(range(3))

STRATS = {
    "linear": ["local", "centralized", "fedavg", "fedprox", "oracle", "flexlora", "ffa", "fedsa",
               "fedsaa", "fedsaa_r0", "fedsaa_u"],
    "nonlinear": ["local", "centralized", "fedavg", "fedprox", "oracle", "ffa", "fedsa"],
    "none": ["probe"],
}
PRODUCT = {"flexlora", "fedsaa", "fedsaa_r0", "fedsaa_u"}


# ------------------------------------------------------------------ data
class Split:
    def __init__(self, Xs, ys):
        self.C = len(Xs)
        self.n = np.array([len(y) for y in ys])
        N, d = self.n.max(), Xs[0].shape[1]
        self.X = np.zeros((self.C, N, d), np.float32)
        self.y = np.zeros((self.C, N), np.int64)
        self.valid = np.zeros((self.C, N), bool)
        for c, (X, y) in enumerate(zip(Xs, ys)):
            self.X[c, :len(y)], self.y[c, :len(y)], self.valid[c, :len(y)] = X, y, True


def load_preset(path):
    z = np.load(path, allow_pickle=False)
    meta = json.loads(str(z["meta"]))
    C = meta["n_clients"]
    sp = {k: Split([z[f"X{k}_{c}"] for c in range(C)], [z[f"y{k}_{c}"] for c in range(C)])
          for k in ("tr", "va", "te")}
    return meta, sp


# ------------------------------------------------------------------ model
def init_params(rngs, C, d, K, adapter):
    """One initialization per replica (its own seed), shared by all clients of that replica."""
    W = np.stack([r.normal(0, 0.01, (K, d)) for r in rngs]).astype(np.float32)
    P = {"W": np.repeat(W[:, None], C, 1), "c": np.zeros((len(rngs), C, K), np.float32)}
    if adapter != "none":
        A = np.stack([r.normal(0, 1 / math.sqrt(d), (RANK, d)) for r in rngs]).astype(np.float32)
        P["A"] = np.repeat(A[:, None], C, 1)
        P["B"] = np.zeros((len(rngs), C, d, RANK), np.float32)
    return P


def T(x):
    return np.swapaxes(x, -1, -2)


def forward(P, X, kmask, adapter):
    """X: (R, C, b, d) or (C, b, d), broadcast over replicas.

    logits = W h + c with h = x + s B u, computed as  x W^T + s u (W B)^T  so that the (b, d) array h is
    never formed. Returns the logits and a cache for the backward pass."""
    s = np.float32(ALPHA / RANK)
    logits = np.matmul(X, T(P["W"]))
    z = u = WB = None
    if adapter != "none":
        z = np.matmul(X, T(P["A"]))                                     # (R, C, b, r)
        u = z if adapter == "linear" else np.maximum(z, 0)
        WB = np.matmul(P["W"], P["B"])                                  # (R, C, K, r)
        logits += s * np.matmul(u, T(WB))
    logits += P["c"][..., None, :]
    logits = np.where(kmask[None, :, None, :], logits, np.float32(-1e9))
    return logits, (X, z, u, WB)


def softmax(lg):
    lg = lg - lg.max(-1, keepdims=True)
    e = np.exp(lg)
    return e / e.sum(-1, keepdims=True)


def loss_and_grads(P, X, y, wrow, kmask, adapter):
    """Mean cross-entropy per (replica, client), rows weighted by wrow, and its gradients."""
    logits, (X, z, u, WB) = forward(P, X, kmask, adapter)
    sh = logits - logits.max(-1, keepdims=True)
    lse = np.log(np.exp(sh).sum(-1))
    loss = -((np.take_along_axis(sh, y[..., None], -1)[..., 0] - lse) * wrow).sum(-1)
    p = softmax(logits)
    py = np.take_along_axis(p, y[..., None], -1)[..., 0]
    g = p
    np.put_along_axis(g, y[..., None], py[..., None] - 1.0, -1)
    g *= wrow[..., None]
    gT = T(g)                                                            # (R, C, K, b)
    G = {"W": np.matmul(gT, X), "c": g.sum(-2)}
    if adapter != "none":
        s = np.float32(ALPHA / RANK)
        gu = np.matmul(gT, u)                                            # (R, C, K, r)
        G["W"] += s * np.matmul(gu, T(P["B"]))                           # g^T (s u B^T)
        G["B"] = s * np.matmul(T(P["W"]), gu)                            # s W^T g^T u
        Gu = s * np.matmul(g, WB)                                        # (R, C, b, r)
        Gz = Gu if adapter == "linear" else Gu * (z > 0)
        G["A"] = np.matmul(T(Gz), X)                                     # (R, C, r, d)
    return loss, G


class Adam:
    """Adam over (replica, client)-stacked parameters; rows whose mask is False are left untouched."""

    def __init__(self, P, lr, b1=0.9, b2=0.999, eps=1e-8):
        self.lr = np.asarray(lr, np.float32).reshape(-1, 1)            # one learning rate per replica
        self.b1, self.b2, self.eps = b1, b2, eps
        self.m = {k: np.zeros_like(v) for k, v in P.items()}
        self.v = {k: np.zeros_like(v) for k, v in P.items()}
        self.t = {k: np.zeros(v.shape[:2], np.float32) for k, v in P.items()}

    def step(self, P, G, masks):
        for k, g in G.items():
            mk = masks[k]
            if not mk.any():
                continue
            ex = (1,) * (g.ndim - 2)
            full = bool(mk.all())
            self.t[k] += mk
            m, v = self.m[k], self.v[k]
            if full:
                m *= self.b1; m += (1 - self.b1) * g
                g *= g
                v *= self.b2; v += (1 - self.b2) * g
            else:
                mm = mk.reshape(mk.shape + ex).astype(np.float32)
                m += mm * ((1 - self.b1) * (g - m))
                v += mm * ((1 - self.b2) * (g * g - v))
            t = np.maximum(self.t[k], 1).reshape(mk.shape + ex)
            step = (self.lr.reshape(self.lr.shape + ex) / (1 - self.b1 ** t)).astype(np.float32)
            den = np.sqrt(v * (1 / (1 - self.b2 ** t)).astype(np.float32))
            den += self.eps
            upd = np.divide(m, den)
            upd *= step
            if not full:
                upd *= mk.reshape(mk.shape + ex)
            P[k] -= upd


# ------------------------------------------------------------------ server-side aggregation
def factors_from(U):
    """Rank-r truncated SVD of each matrix in U (C, d, d); returns A', B' and retained energy."""
    Pm, S, Qt = np.linalg.svd(U.astype(np.float32))
    sq = np.sqrt(S[:, :RANK])
    B = Pm[:, :, :RANK] * sq[:, None, :]
    A = sq[:, :, None] * Qt[:, :RANK, :]
    energy = (S[:, :RANK] ** 2).sum(1) / np.maximum((S ** 2).sum(1), 1e-30)
    return A.astype(np.float32), B.astype(np.float32), energy


def group_mean(x, w, groups):
    """x: (R, C, ...); weighted mean over the clients of each group, per replica."""
    out = np.empty_like(x)
    for g in np.unique(groups):
        idx = groups == g
        ww = (w[idx] / w[idx].sum()).astype(np.float32)
        out[:, idx] = np.tensordot(x[:, idx], ww, axes=([1], [0]))[:, None]
    return out


def aggregate(strategy, P, n, spec, logs):
    R, C = P["A"].shape[:2]
    allg = np.zeros(C, int)
    w = n.astype(np.float64)
    if strategy in ("fedavg", "fedprox"):
        P["A"], P["B"] = group_mean(P["A"], w, allg), group_mean(P["B"], w, allg)
    elif strategy == "oracle":
        P["A"], P["B"] = group_mean(P["A"], w, spec), group_mean(P["B"], w, spec)
    elif strategy == "ffa":
        P["B"] = group_mean(P["B"], w, allg)
    elif strategy == "fedsa":
        P["A"] = group_mean(P["A"], w, allg)
    elif strategy == "flexlora":
        for r in range(R):
            U = np.matmul(P["B"][r], P["A"][r]).astype(np.float64)
            G = np.tensordot(w / w.sum(), U, axes=1)
            A, B, e = factors_from(G[None])
            P["A"][r], P["B"][r] = A[0], B[0]
            logs[r]["energy"] = np.repeat(e, C)
    elif strategy.startswith("fedsaa"):
        rho = 0.0 if strategy == "fedsaa_r0" else RHO
        for r in range(R):
            U = np.matmul(P["B"][r], P["A"][r])                             # (C, d, d)
            flat = U.reshape(C, -1)
            nrm = np.linalg.norm(flat, axis=1)
            S = (flat @ flat.T) / np.maximum(np.outer(nrm, nrm), 1e-12)
            lg = S / TAU
            Wt = np.exp(lg - lg.max(1, keepdims=True))
            Wt /= Wt.sum(1, keepdims=True)
            Ut = (1 - LAM) * np.tensordot(Wt, U, axes=1) + LAM * U.mean(0, keepdims=True)
            if strategy == "fedsaa_u":
                Ut = (1 - rho) * Ut + rho * U
            A, B, e = factors_from(Ut)
            if strategy == "fedsaa":
                A = (1 - rho) * A + rho * P["A"][r]
                B = (1 - rho) * B + rho * P["B"][r]
            P["A"][r], P["B"][r] = A, B
            logs[r].update(S=S, Wt=Wt, energy=e)


# ------------------------------------------------------------------ evaluation
def predict_proba(P, split, kmask, adapter):
    logits, _ = forward(P, split.X, kmask, adapter)
    return softmax(logits)


def macro_f1(y, yp, labels):
    f = []
    for k in labels:
        tp = np.sum((yp == k) & (y == k)); fp = np.sum((yp == k) & (y != k)); fn = np.sum((yp != k) & (y == k))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))


def auc_ovr(y, p, K):
    from scipy.stats import rankdata
    out = []
    for k in range(K):
        pos = y == k
        npos, nneg = pos.sum(), (~pos).sum()
        if npos == 0 or nneg == 0:
            continue
        r = rankdata(p[:, k])
        out.append((r[pos].sum() - npos * (npos + 1) / 2) / (npos * nneg))
    return float(np.mean(out)) if out else float("nan")


def full_metrics(prob, split, nclass):
    """prob: (C, N, K) for one replica."""
    acc, f1, auc, correct = [], [], [], []
    for c in range(split.C):
        n = split.n[c]
        y, pr = split.y[c, :n], prob[c, :n, :nclass[c]]
        yp = pr.argmax(1)
        ok = yp == y
        correct.append(ok)
        acc.append(float(ok.mean()))
        f1.append(macro_f1(y, yp, np.union1d(np.unique(y), np.unique(yp))))
        auc.append(auc_ovr(y, pr, nclass[c]))
    acc = np.array(acc)
    return {"acc_eq": float(acc.mean()), "acc_w": float(np.concatenate(correct).mean()),
            "f1": float(np.nanmean(f1)), "auc": float(np.nanmean(auc)), "worst": float(acc.min()),
            "per_client": acc.tolist(), "f1_client": f1, "auc_client": auc}, correct


def accs(prob, split):
    """prob: (R, C, N, K) -> equal-weight accuracy per replica."""
    ok = (prob.argmax(-1) == split.y[None]) & split.valid[None]
    return (ok.sum(-1) / split.n[None]).mean(-1)


# ------------------------------------------------------------------ one run
def run_batch(meta, sp, adapter, strategy, replicas, rounds=ROUNDS, checkpoints=CHECKPOINTS, keep=True):
    """Train one strategy for several replicas at once; replicas = [(seed, lr, mu), ...]."""
    tr, va, te = sp["tr"], sp["va"], sp["te"]
    C, d = tr.C, tr.X.shape[2]
    R = len(replicas)
    nclass = np.array(meta["n_classes"])
    K = int(nclass.max())
    kmask = np.arange(K)[None, :] < nclass[:, None]
    spec = np.array(meta["client_specialty"])
    rngs = [np.random.default_rng(seed) for seed, _, _ in replicas]
    lr = np.array([x[1] for x in replicas], np.float32)
    mu = np.array([x[2] for x in replicas], np.float32)
    P = init_params(rngs, C, d, K, adapter)
    trainable = {k: np.ones((R, C), bool) for k in P}
    if strategy == "ffa":
        trainable["A"][:] = False
    tied = strategy == "centralized"
    steps = int(math.ceil(tr.n.max() / BATCH))
    cidx = np.arange(C)[None, :, None]
    outs = [{"test_curve": [], "val_curve": [], "energy_curve": [], "checkpoints": {}, "logs": {}} for _ in range(R)]
    corrects = [{} for _ in range(R)]
    for t in range(1, rounds + 1):
        glob = {k: P[k].copy() for k in ("A", "B")} if strategy == "fedprox" else None
        opt = Adam(P, lr)
        for _ in range(LOCAL_EPOCHS):
            perm = np.zeros((R, C, steps * BATCH), np.int64)
            for r in range(R):
                for c in range(C):
                    perm[r, c, :tr.n[c]] = rngs[r].permutation(tr.n[c])
            for s in range(steps):
                lo = s * BATCH
                cnt = np.clip(tr.n - lo, 0, BATCH)                       # (C,)
                active_c = cnt > 0
                if not active_c.any():
                    break
                idx = perm[:, :, lo:lo + BATCH]
                val = (np.arange(BATCH)[None, :] < cnt[:, None])          # (C, b)
                Xb = tr.X[cidx, idx]                                      # (R, C, b, d)
                yb = tr.y[cidx, idx]
                wrow = np.broadcast_to((val / np.maximum(cnt, 1)[:, None]).astype(np.float32), (R, C, BATCH))
                _, G = loss_and_grads(P, Xb, yb, wrow, kmask, adapter)
                active = np.broadcast_to(active_c, (R, C))
                if glob is not None:
                    for k in ("A", "B"):
                        G[k] += mu.reshape(R, 1, 1, 1) * (P[k] - glob[k])
                if tied:
                    masks = {}
                    am = active_c.astype(np.float32)
                    for k in list(G):
                        ex = (1,) * (G[k].ndim - 2)
                        if k in ("A", "B"):
                            G[k] = np.repeat((G[k] * am.reshape((1, C) + ex)).sum(1, keepdims=True) / am.sum(), C, 1)
                            masks[k] = np.ones((R, C), bool)
                        else:
                            Gk = np.zeros_like(G[k])
                            mk = np.zeros((R, C), bool)
                            for g in np.unique(spec):
                                gi = spec == g
                                ga = gi & active_c
                                if ga.any():
                                    Gk[:, gi] = G[k][:, ga].mean(1, keepdims=True)
                                    mk[:, gi] = True
                            G[k], masks[k] = Gk, mk
                else:
                    masks = {k: active for k in G}
                masks = {k: masks[k] & trainable[k] for k in masks}
                opt.step(P, G, masks)
        logs = [{} for _ in range(R)]
        if adapter != "none" and strategy not in ("local", "centralized", "probe"):
            aggregate(strategy, P, tr.n, spec, logs)
        pte = predict_proba(P, te, kmask, adapter)
        a_te, a_va = accs(pte, te), accs(predict_proba(P, va, kmask, adapter), va)
        if not np.all(np.isfinite(a_te)):
            raise FloatingPointError("non-finite accuracy")
        for r in range(R):
            o = outs[r]
            o["test_curve"].append(float(a_te[r]))
            o["val_curve"].append(float(a_va[r]))
            if "energy" in logs[r]:
                o["energy_curve"].append(float(np.mean(logs[r]["energy"])))
            if t in LOG_ROUNDS and logs[r]:
                o["logs"][t] = {k: np.asarray(v).round(5).tolist() for k, v in logs[r].items()}
            if t in checkpoints:
                m, corr = full_metrics(pte[r], te, nclass)
                m["val_acc_eq"] = float(a_va[r])
                o["checkpoints"][t] = m
                if keep:
                    corrects[r][t] = corr
    return outs, corrects


# ------------------------------------------------------------------ orchestration
def key_of(stage, preset, adapter, strategy, seed, lr, mu):
    return f"{stage}|{preset}|{adapter}|{strategy}|{seed}|{lr:g}|{mu:g}"


def done_keys(path):
    if not os.path.exists(path):
        return set()
    with open(path) as fh:
        return {json.loads(l)["key"] for l in fh if l.strip()}


def tune_jobs(presets, adapters):
    jobs = []
    for preset in presets:
        for adapter, strats in STRATS.items():
            if adapter not in adapters:
                continue
            for st in strats:
                for lr in LR_GRID:
                    for mu in (MU_GRID if st == "fedprox" else (0.0,)):
                        jobs.append((preset, adapter, st, TUNE_SEED, lr, mu))
    return jobs


def select_configs(path):
    """Best lr (and mu) per preset x adapter x strategy: mean validation accuracy at the checkpoints."""
    best = {}
    with open(path) as fh:
        for l in fh:
            r = json.loads(l)
            if r["stage"] != "tune":
                continue
            score = float(np.mean([r["checkpoints"][str(t)]["val_acc_eq"] for t in TUNE_CHECKPOINTS]))
            k = (r["preset"], r["adapter"], r["strategy"])
            if k not in best or score > best[k][0]:
                best[k] = (score, r["lr"], r["mu"])
    return best


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--feats", default="feats")
    ap.add_argument("--out", default="results")
    ap.add_argument("--stage", choices=["tune", "main"], required=True)
    ap.add_argument("--budget-min", type=float, default=40.0)
    ap.add_argument("--presets", default="A,B")
    ap.add_argument("--seeds", type=int, default=len(MAIN_SEEDS))
    ap.add_argument("--adapters", default="linear,nonlinear,none")
    args = ap.parse_args()
    os.makedirs(os.path.join(args.out, "runs"), exist_ok=True)
    jl = os.path.join(args.out, "runs.jsonl")
    presets = args.presets.split(",")
    adapters = args.adapters.split(",")
    data = {p: load_preset(os.path.join(args.feats, f"preset_{p}.npz")) for p in presets}
    for p, (meta, _) in data.items():                 # keep the client metadata with the results
        with open(os.path.join(args.out, f"meta_{p}.json"), "w") as fh:
            json.dump(meta, fh)

    if args.stage == "tune":
        jobs = [("tune",) + j for j in tune_jobs(presets, adapters)]
    else:
        tj = [j for j in tune_jobs(presets, adapters)]
        have = done_keys(jl)
        missing = [j for j in tj if key_of("tune", *j) not in have]
        if missing:
            raise SystemExit(f"tuning incomplete: {len(missing)} tuning runs missing; run --stage tune first")
        best = select_configs(jl)
        with open(os.path.join(args.out, "selected_configs.json"), "w") as fh:
            json.dump({"|".join(k): {"val_score": v[0], "lr": v[1], "mu": v[2]} for k, v in best.items()}, fh, indent=1)
        jobs = []
        for preset in presets:
            for adapter, strats in STRATS.items():
                if adapter not in adapters:
                    continue
                for st in strats:
                    _, lr, mu = best[(preset, adapter, st)]
                    for seed in MAIN_SEEDS[:args.seeds]:
                        jobs.append(("main", preset, adapter, st, seed, lr, mu))

    have = done_keys(jl)
    todo = [j for j in jobs if key_of(*j) not in have]
    print(f"stage {args.stage}: {len(jobs)} runs, {len(jobs) - len(todo)} done, {len(todo)} to do", flush=True)
    batches = {}
    for j in todo:                                   # one batch per preset x adapter x strategy
        batches.setdefault(j[:4], []).append(j)
    t_start, done = time.time(), 0
    for (stage, preset, adapter, st), js in batches.items():
        if (time.time() - t_start) / 60 > args.budget_min:
            print(f"time budget reached; {len(todo) - done} runs left. Run the same command again.", flush=True)
            return
        t0 = time.time()
        meta, sp = data[preset]
        rounds, cps = (ROUNDS, CHECKPOINTS) if stage == "main" else (TUNE_ROUNDS, TUNE_CHECKPOINTS)
        outs, corrs = run_batch(meta, sp, adapter, st, [(j[4], j[5], j[6]) for j in js], rounds=rounds,
                                checkpoints=cps, keep=(stage == "main"))
        sec = (time.time() - t0) / len(js)
        with open(jl, "a") as fh:
            for j, res, corr in zip(js, outs, corrs):
                key = key_of(*j)
                rec = {"key": key, "stage": stage, "preset": preset, "adapter": adapter, "strategy": st,
                       "seed": j[4], "lr": j[5], "mu": j[6], "rounds": rounds, "local_epochs": LOCAL_EPOCHS,
                       "rank": RANK, "alpha": ALPHA, "tau": TAU, "lam": LAM, "rho": RHO, "sec": sec,
                       "checkpoints": {str(t): v for t, v in res["checkpoints"].items()},
                       "test_curve": res["test_curve"], "val_curve": res["val_curve"],
                       "energy_curve": res["energy_curve"] or None}
                if stage == "main":
                    fn = os.path.join(args.out, "runs", key.replace("|", "__") + ".npz")
                    payload = {f"correct_{t}_{c}": v for t, cl in corr.items() for c, v in enumerate(cl)}
                    payload["logs"] = np.array(json.dumps({str(t): v for t, v in res["logs"].items()}))
                    np.savez_compressed(fn, **payload)
                fh.write(json.dumps(rec) + "\n")
                done += 1
        accs_ = [o["checkpoints"][cps[-1]]["acc_eq"] for o in outs]
        print(f"[{done}/{len(todo)}] {stage} {preset} {adapter} {st}: {len(js)} runs, "
              f"test acc@{cps[-1]} {np.mean(accs_):.4f}, {sec:.0f} s/run", flush=True)
    print(f"stage {args.stage} complete", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile fedbench_analyze.py
#!/usr/bin/env python3
"""Stage 3: every table, test and figure for the revised manuscript, from the raw run records.

Usage:
    python fedbench_analyze.py --feats feats --results results --out analysis

Writes CSV tables, PNG/PDF figures and summary.md to --out. Nothing is typed in by hand.
Seed-level tests: paired t-test, Wilcoxon signed-rank and exact sign-flip test over the 10 seeds, with
Holm correction within each family. Image-level tests: per-image correctness averaged over seeds,
client-equal-weight mean difference, stratified bootstrap CI and sign-flip randomization p value.
"""
import argparse
import glob
import itertools
import json
import os

import numpy as np
import pandas as pd
from scipy import stats

LABEL = {"local": "Local-only", "centralized": "Centralized", "fedavg": "FedAvg", "fedprox": "FedProx",
         "oracle": "Specialty oracle", "flexlora": "FlexLoRA", "ffa": "FFA-LoRA", "fedsa": "FedSA-LoRA",
         "fedsaa": "FedSAA", "fedsaa_r0": "FedSAA (rho = 0)", "fedsaa_u": "FedSAA-U", "probe": "Linear probe"}
PLANNED = [("fedavg", "local"), ("fedprox", "local"), ("oracle", "local"), ("flexlora", "local"),
           ("ffa", "local"), ("fedsa", "local"), ("fedsaa", "local"), ("fedsaa_r0", "local"), ("fedsaa_u", "local"),
           ("centralized", "local"), ("fedsaa", "fedavg"), ("fedsaa_u", "fedavg"), ("fedsaa_u", "fedsaa"),
           ("fedsaa_r0", "fedsaa"), ("fedsaa_u", "fedsaa_r0"), ("fedsaa_u", "flexlora"), ("oracle", "fedavg"),
           ("fedsaa_u", "oracle"), ("fedprox", "fedavg")]
METRICS = ["acc_eq", "acc_w", "f1", "auc", "worst"]


def holm(p):
    p = np.asarray(p, float)
    out = np.full(len(p), np.nan)
    ok = ~np.isnan(p)
    q = p[ok]
    order = np.argsort(q)
    adj, run = np.empty(len(q)), 0.0
    for rank, i in enumerate(order):
        run = max(run, (len(q) - rank) * q[i])
        adj[i] = min(1.0, run)
    out[ok] = adj
    return out


def signflip_p(d):
    d = np.asarray(d, float)
    n = len(d)
    if n <= 16:
        signs = np.array(list(itertools.product([1, -1], repeat=n)))
        stat = np.abs((signs * d).mean(1))
    else:
        rng = np.random.default_rng(0)
        stat = np.abs((rng.choice([1, -1], (20000, n)) * d).mean(1))
    return float(np.mean(stat >= abs(d.mean()) - 1e-12))


def paired(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    n = len(d)
    mu, sd = d.mean(), d.std(ddof=1)
    se = sd / np.sqrt(n)
    if se > 0:
        tval = mu / se
        p_t = 2 * stats.t.sf(abs(tval), n - 1)
    else:
        tval, p_t = np.nan, (1.0 if mu == 0 else 0.0)
    crit = stats.t.ppf(0.975, n - 1)
    try:
        p_w = float(stats.wilcoxon(d).pvalue) if np.any(d != 0) else 1.0
    except ValueError:
        p_w = np.nan
    return {"n": n, "diff": mu, "sd": sd, "ci_lo": mu - crit * se, "ci_hi": mu + crit * se, "t": tval,
            "p_t": p_t, "p_wilcoxon": p_w, "p_signflip": signflip_p(d), "same_sign": int(max((d > 0).sum(), (d < 0).sum()))}


def load(results):
    rows = [json.loads(l) for l in open(os.path.join(results, "runs.jsonl")) if l.strip()]
    main = [r for r in rows if r["stage"] == "main"]
    tune = [r for r in rows if r["stage"] == "tune"]
    return main, tune


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--feats", default="feats")
    ap.add_argument("--results", default="results")
    ap.add_argument("--out", default="analysis")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    main_runs, tune_runs = load(args.results)
    meta = {}
    for f in sorted(glob.glob(os.path.join(args.results, "meta_*.json"))):
        meta[os.path.basename(f)[5:-5]] = json.load(open(f))
    if not meta:
        for f in sorted(glob.glob(os.path.join(args.feats, "preset_*.npz"))):
            meta[os.path.basename(f)[7:-4]] = json.loads(str(np.load(f, allow_pickle=False)["meta"]))
    cps = sorted({int(t) for r in main_runs for t in r["checkpoints"]})
    R = lambda p, a, s: sorted([r for r in main_runs if (r["preset"], r["adapter"], r["strategy"]) == (p, a, s)],
                               key=lambda r: r["seed"])
    combos = sorted({(r["preset"], r["adapter"], r["strategy"]) for r in main_runs})
    W = lambda name: os.path.join(args.out, name)

    # ---------------- main table
    t_main = []
    for p, a, s in combos:
        rs = R(p, a, s)
        for t in cps:
            row = {"preset": p, "adapter": a, "strategy": s, "round": t, "n_seeds": len(rs),
                   "lr": rs[0]["lr"], "mu": rs[0]["mu"]}
            for m in METRICS:
                v = np.array([r["checkpoints"][str(t)][m] for r in rs]) * (100 if m in ("acc_eq", "acc_w", "worst") else 1)
                row[f"{m}_mean"], row[f"{m}_sd"] = v.mean(), v.std(ddof=1)
            t_main.append(row)
    t_main = pd.DataFrame(t_main)
    t_main.to_csv(W("T_main.csv"), index=False)

    # ---------------- seed-level contrasts
    t_pairs = []
    for p in sorted({c[0] for c in combos}):
        for a in sorted({c[1] for c in combos if c[0] == p}):
            strs = [c[2] for c in combos if c[:2] == (p, a)]
            if a == "none":
                continue
            for t in cps:
                for m in METRICS:
                    fam = []
                    for x, y in itertools.combinations(strs, 2):
                        if (y, x) in PLANNED:
                            x, y = y, x
                        sc = 100 if m in ("acc_eq", "acc_w", "worst") else 1
                        res = paired([r["checkpoints"][str(t)][m] * sc for r in R(p, a, x)],
                                     [r["checkpoints"][str(t)][m] * sc for r in R(p, a, y)])
                        res.update(preset=p, adapter=a, round=t, metric=m, a=x, b=y, planned=(x, y) in PLANNED)
                        fam.append(res)
                    ph = holm([f["p_t"] for f in fam])
                    pw = holm([f["p_wilcoxon"] for f in fam])
                    pl = [i for i, f in enumerate(fam) if f["planned"]]
                    php = holm([fam[i]["p_t"] for i in pl])
                    for i, f in enumerate(fam):
                        f["p_t_holm_all"], f["p_w_holm_all"] = ph[i], pw[i]
                        f["p_t_holm_planned"] = php[pl.index(i)] if i in pl else np.nan
                    t_pairs.extend(fam)
        # adapter-level contrasts (Proposition 1): linear and nonlinear adapters against the linear probe
        for t in cps:
            for x in ("linear", "nonlinear"):
                if R(p, x, "local") and R(p, "none", "probe"):
                    res = paired([r["checkpoints"][str(t)]["acc_eq"] * 100 for r in R(p, x, "local")],
                                 [r["checkpoints"][str(t)]["acc_eq"] * 100 for r in R(p, "none", "probe")])
                    res.update(preset=p, adapter=f"{x} vs none", round=t, metric="acc_eq", a=f"{x}:local",
                               b="none:probe", planned=True)
                    t_pairs.append(res)
            for s in ("local", "fedavg", "oracle", "centralized"):
                if R(p, "linear", s) and R(p, "nonlinear", s):
                    res = paired([r["checkpoints"][str(t)]["acc_eq"] * 100 for r in R(p, "nonlinear", s)],
                                 [r["checkpoints"][str(t)]["acc_eq"] * 100 for r in R(p, "linear", s)])
                    res.update(preset=p, adapter="nonlinear vs linear", round=t, metric="acc_eq",
                               a=f"nonlinear:{s}", b=f"linear:{s}", planned=True)
                    t_pairs.append(res)
    t_pairs = pd.DataFrame(t_pairs)
    t_pairs.to_csv(W("T_pairs_seed.csv"), index=False)

    # ---------------- image-level contrasts (planned only)
    def correct(run, t, C):
        fn = os.path.join(args.results, "runs", run["key"].replace("|", "__") + ".npz")
        z = np.load(fn, allow_pickle=False)
        return [z[f"correct_{t}_{c}"].astype(float) for c in range(C)]

    t_img = []
    rng = np.random.default_rng(0)
    for p in sorted({c[0] for c in combos}):
        C = meta[p]["n_clients"]
        for a in ("linear", "nonlinear"):
            for x, y in PLANNED:
                rx, ry = R(p, a, x), R(p, a, y)
                if not rx or not ry:
                    continue
                for t in cps:
                    cx = [correct(r, t, C) for r in rx]
                    cy = [correct(r, t, C) for r in ry]
                    dcl = [np.mean([s[c] for s in cx], 0) - np.mean([s[c] for s in cy], 0) for c in range(C)]
                    stat = np.mean([d.mean() for d in dcl]) * 100
                    boot = np.array([np.mean([d[rng.integers(0, len(d), len(d))].mean() for d in dcl])
                                     for _ in range(1000)]) * 100
                    flips = np.array([np.mean([(d * rng.choice([-1, 1], len(d))).mean() for d in dcl])
                                      for _ in range(1000)]) * 100
                    wdiff = np.concatenate(dcl).mean() * 100
                    t_img.append({"preset": p, "adapter": a, "round": t, "a": x, "b": y, "diff_eq": stat,
                                  "ci_lo": np.percentile(boot, 2.5), "ci_hi": np.percentile(boot, 97.5),
                                  "p_signflip_images": float((np.abs(flips) >= abs(stat) - 1e-12).mean()),
                                  "diff_size_weighted": wdiff})
    t_img = pd.DataFrame(t_img)
    if len(t_img):
        for (p, a, t), g in t_img.groupby(["preset", "adapter", "round"]):
            t_img.loc[g.index, "p_holm"] = holm(g.p_signflip_images.values)
    t_img.to_csv(W("T_pairs_image.csv"), index=False)

    # ---------------- per-specialty and per-client gains over local
    t_spec, t_cli = [], []
    for p, a, s in combos:
        if s in ("local", "probe"):
            continue
        rl, rs = R(p, a, "local"), R(p, a, s)
        sp = np.array(meta[p]["client_specialty"])
        for t in cps:
            loc = np.array([r["checkpoints"][str(t)]["per_client"] for r in rl]) * 100
            cur = np.array([r["checkpoints"][str(t)]["per_client"] for r in rs]) * 100
            for g, name in enumerate(meta[p]["specialties"]):
                d = cur[:, sp == g].mean(1) - loc[:, sp == g].mean(1)
                res = paired(d, np.zeros_like(d))
                t_spec.append({"preset": p, "adapter": a, "strategy": s, "round": t, "specialty": name,
                               "gain": res["diff"], "ci_lo": res["ci_lo"], "ci_hi": res["ci_hi"], "p_t": res["p_t"]})
            dc = (cur - loc).mean(0)
            t_cli.append({"preset": p, "adapter": a, "strategy": s, "round": t, "clients_above": int((dc > 0).sum()),
                          "clients_below": int((dc < 0).sum()), "median_gain": float(np.median(dc)),
                          "min_gain": float(dc.min()), "max_gain": float(dc.max())})
    pd.DataFrame(t_spec).to_csv(W("T_specialty_gain.csv"), index=False)
    pd.DataFrame(t_cli).to_csv(W("T_client_gain.csv"), index=False)

    # ---------------- convergence
    t_conv = []
    for p, a, s in combos:
        rs = R(p, a, s)
        cur = np.array([r["test_curve"] for r in rs]) * 100
        T = cur.shape[1]
        mean = cur.mean(0)
        last = min(20, T)
        slope = np.mean([np.polyfit(np.arange(T - last, T) + 1, c[-last:], 1)[0] for c in cur])
        within = int(np.argmax(np.abs(mean - mean[-1]) <= 0.5)) + 1
        t_conv.append({"preset": p, "adapter": a, "strategy": s, "final": mean[-1], "peak": mean.max(),
                       "peak_round": int(mean.argmax()) + 1, "slope_last20_per_round": slope,
                       "first_round_within_0.5_of_final": within,
                       **{f"r{k}": mean[k - 1] for k in (10, 20, 40, 100, 150, 200) if k <= T}})
    pd.DataFrame(t_conv).to_csv(W("T_convergence.csv"), index=False)

    # ---------------- measured similarities, attention shares and SVD energy
    t_sim = []
    for p, a, s in combos:
        if not s.startswith("fedsaa") and s != "flexlora":
            continue
        sp = np.array(meta[p]["client_specialty"])
        same = (sp[:, None] == sp[None, :]) & ~np.eye(len(sp), dtype=bool)
        other = sp[:, None] != sp[None, :]
        acc = {}
        for r in R(p, a, s):
            fn = os.path.join(args.results, "runs", r["key"].replace("|", "__") + ".npz")
            logs = json.loads(str(np.load(fn, allow_pickle=False)["logs"]))
            for t, lg in logs.items():
                e = acc.setdefault(int(t), {k: [] for k in ("S_same", "S_other", "w_self", "w_same", "w_other", "energy")})
                if "S" in lg:
                    S, Wt = np.array(lg["S"]), np.array(lg["Wt"])
                    e["S_same"].append(S[same].mean()); e["S_other"].append(S[other].mean())
                    e["w_self"].append(np.diag(Wt).mean())
                    e["w_same"].append((Wt * same).sum(1).mean()); e["w_other"].append((Wt * other).sum(1).mean())
                if "energy" in lg:
                    e["energy"].append(float(np.mean(lg["energy"])))
        for t, e in sorted(acc.items()):
            t_sim.append({"preset": p, "adapter": a, "strategy": s, "round": t,
                          **{k: (float(np.mean(v)) if v else np.nan) for k, v in e.items()}})
    pd.DataFrame(t_sim).to_csv(W("T_similarity_energy.csv"), index=False)

    # ---------------- tuning record
    t_tune = pd.DataFrame([{"preset": r["preset"], "adapter": r["adapter"], "strategy": r["strategy"], "lr": r["lr"],
                            "mu": r["mu"], **{f"val_r{t}": v["val_acc_eq"] * 100 for t, v in r["checkpoints"].items()}}
                           for r in tune_runs])
    t_tune.to_csv(W("T_tuning.csv"), index=False)
    sel = os.path.join(args.results, "selected_configs.json")
    if os.path.exists(sel):
        with open(sel) as fh, open(W("selected_configs.json"), "w") as gh:
            gh.write(fh.read())

    figures(main_runs, combos, meta, cps, args.out, pd.DataFrame(t_spec))
    summary(t_main, t_pairs, t_img, pd.DataFrame(t_conv), pd.DataFrame(t_sim), cps, W("summary.md"))
    print("analysis written to", args.out)


def figures(main_runs, combos, meta, cps, out, t_spec):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 7, "axes.spines.top": False, "axes.spines.right": False, "pdf.fonttype": 42})
    col = {"local": "#0072B2", "centralized": "#000000", "fedavg": "#E69F00", "fedprox": "#CC79A7",
           "oracle": "#009E73", "flexlora": "#56B4E9", "ffa": "#999999", "fedsa": "#F0E442",
           "fedsaa": "#D55E00", "fedsaa_r0": "#8C2D04", "fedsaa_u": "#7B3294", "probe": "#666666"}
    presets = sorted({c[0] for c in combos})
    for a in ("linear", "nonlinear"):
        fig, axes = plt.subplots(len(presets), 1, figsize=(3.4, 2.2 * len(presets)), sharex=True, squeeze=False)
        for ax, p in zip(axes[:, 0], presets):
            for (pp, aa, s) in combos:
                if (pp, aa) != (p, a):
                    continue
                cur = np.array([r["test_curve"] for r in main_runs if (r["preset"], r["adapter"], r["strategy"], r["stage"]) == (p, a, s, "main")]) * 100
                ax.plot(np.arange(1, cur.shape[1] + 1), cur.mean(0), color=col[s], lw=1.0, label=LABEL[s])
            ax.set_title(f"Preset {p}, {a} adapter", loc="left", fontsize=7.5, fontweight="bold")
            ax.set_ylabel("Test accuracy (%)")
            ax.grid(axis="y", lw=0.3, alpha=0.5)
        axes[-1, 0].set_xlabel("Communication round")
        axes[0, 0].legend(frameon=False, fontsize=5.5, ncol=2)
        fig.savefig(os.path.join(out, f"fig_curves_{a}.png"), dpi=600, bbox_inches="tight")
        fig.savefig(os.path.join(out, f"fig_curves_{a}.pdf"), bbox_inches="tight")
        plt.close(fig)
    if len(t_spec):
        t = max(cps)
        sub = t_spec[(t_spec["round"] == t) & (t_spec.adapter == "linear") &
                     t_spec.strategy.isin(["fedavg", "oracle", "fedsaa", "fedsaa_u"])]
        names = list(dict.fromkeys(sub.preset + ":" + sub.specialty))
        fig, ax = plt.subplots(figsize=(3.4, 3.6))
        off = {"fedavg": 0.24, "oracle": 0.08, "fedsaa": -0.08, "fedsaa_u": -0.24}
        for s, g in sub.groupby("strategy"):
            y = [-names.index(n) + off[s] for n in g.preset + ":" + g.specialty]
            ax.errorbar(g.gain, y, xerr=[g.gain - g.ci_lo, g.ci_hi - g.gain], fmt="D", ms=3, lw=0.7,
                        color=col[s], label=LABEL[s])
        ax.axvline(0, color="k", lw=0.7)
        ax.set_yticks([-i for i in range(len(names))])
        ax.set_yticklabels([n.split(":")[1] + f" ({n.split(':')[0]})" for n in names])
        ax.set_xlabel(f"Gain over local-only at round {t} (points), mean and 95% CI")
        ax.legend(frameon=False, fontsize=5.5)
        fig.savefig(os.path.join(out, "fig_specialty_gain.png"), dpi=600, bbox_inches="tight")
        fig.savefig(os.path.join(out, "fig_specialty_gain.pdf"), bbox_inches="tight")
        plt.close(fig)


def summary(t_main, t_pairs, t_img, t_conv, t_sim, cps, path):
    L = ["# Revised experiments: summary", ""]
    for t in cps:
        L.append(f"## Accuracy at round {t} (mean ± SD over seeds, equal-weight / size-weighted)")
        sub = t_main[t_main["round"] == t]
        for _, r in sub.iterrows():
            L.append(f"- {r.preset} {r.adapter:9} {LABEL[r.strategy]:18} {r.acc_eq_mean:6.2f} ± {r.acc_eq_sd:4.2f}"
                     f" / {r.acc_w_mean:6.2f}  F1 {r.f1_mean:.3f}  AUC {r.auc_mean:.4f}  worst {r.worst_mean:5.2f}"
                     f"  (lr {r.lr:g}, mu {r.mu:g})")
        L.append("")
    if len(t_pairs):
        L.append("## Planned seed-level contrasts, accuracy (Holm within the planned family)")
        sub = t_pairs[(t_pairs.metric == "acc_eq") & t_pairs.planned]
        for _, r in sub.iterrows():
            L.append(f"- r{r['round']} {r.preset} {r.adapter}: {r.a} − {r.b} = {r['diff']:+.2f} "
                     f"[{r.ci_lo:+.2f}, {r.ci_hi:+.2f}], p_t {r.p_t:.4f}, Holm {r.p_t_holm_planned:.4f}, "
                     f"Wilcoxon {r.p_wilcoxon:.4f}, same sign {r.same_sign}/{r.n}")
        L.append("")
    if len(t_img):
        L.append("## Image-level contrasts (per-image correctness averaged over seeds)")
        for _, r in t_img.iterrows():
            L.append(f"- r{r['round']} {r.preset} {r.adapter}: {r.a} − {r.b} = {r.diff_eq:+.2f} "
                     f"[{r.ci_lo:+.2f}, {r.ci_hi:+.2f}], p {r.p_signflip_images:.3f}, Holm {r.p_holm:.3f}")
        L.append("")
    L.append("## Convergence")
    for _, r in t_conv.iterrows():
        L.append(f"- {r.preset} {r.adapter} {LABEL[r.strategy]}: final {r.final:.2f}, peak {r.peak:.2f} at round "
                 f"{r.peak_round}, slope over last 20 rounds {r.slope_last20_per_round:+.4f}/round")
    if len(t_sim):
        L.append("")
        L.append("## Measured similarities, attention shares and retained SVD energy")
        for _, r in t_sim.iterrows():
            L.append(f"- {r.preset} {r.adapter} {r.strategy} r{r['round']}: S same {r.S_same:.3f}, S other "
                     f"{r.S_other:.3f}, w self {r.w_self:.3f}, same {r.w_same:.3f}, other {r.w_other:.3f}, "
                     f"energy {r.energy:.3f}")
    with open(path, "w") as fh:
        fh.write("\n".join(L) + "\n")


if __name__ == "__main__":
    main()


In [ ]:
import shutil
os.makedirs("results", exist_ok=True)
if RESUME_FROM and os.path.isdir(RESUME_FROM):
    shutil.copytree(RESUME_FROM, "results", dirs_exist_ok=True)
    print("resumed from", RESUME_FROM)

In [ ]:
!python fedbench_features.py --out feats --root medmnist_data

In [ ]:
!python fedbench_train.py --feats feats --out results --stage tune --budget-min {TUNE_BUDGET_MIN}

In [ ]:
!python fedbench_train.py --feats feats --out results --stage main --budget-min {MAIN_BUDGET_MIN}

In [ ]:
!python fedbench_analyze.py --feats feats --results results --out analysis

In [ ]:
shutil.rmtree("medmnist_data", ignore_errors=True)       # keep the output small
shutil.make_archive("fedbench_v2_results", "zip", ".", "results")
import zipfile
with zipfile.ZipFile("fedbench_v2_results.zip", "a") as z:
    for root, _, files in os.walk("analysis"):
        for f in files:
            z.write(os.path.join(root, f))
print(round(os.path.getsize("fedbench_v2_results.zip") / 1e6, 1), "MB")
print(open("analysis/summary.md").read()[:3000])